In [1]:
from typing import TypedDict, Optional, Union

In [2]:
class State(TypedDict):
    state1: str
    state2: int

state = State(state1="jasdj", state2=2)

In [3]:
square = lambda x: x*x

nums = [1,2,3,4]
squares = list(map(lambda x: square(x), nums))
print(squares)

[1, 4, 9, 16]


In [4]:
### GRAPH 1 ###

from typing import TypedDict, Dict
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    message: str

def greeting_node(agentstate: AgentState) -> AgentState:
    """Nah bruv ts guy bussin fr"""

    agentstate["message"] = "Hey " + agentstate["message"] + "!"
    return agentstate

graph = StateGraph(AgentState)

graph.add_node("greeting", greeting_node)

graph.set_entry_point("greeting")
graph.set_finish_point("greeting")

app = graph.compile()

result = app.invoke({"message": "Bob"})
result["message"]

'Hey Bob!'

In [5]:
### GRAPH 2 ###

from typing import TypedDict, List
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    values: List[int]
    name: str
    result: str

def process_values(state: AgentState) -> AgentState:

    print(state)

    state["result"] = f"Hi there, {state['name']}! Your sum is {sum(state['values'])}"
    print(state)
    return state

graph = StateGraph(AgentState)

graph.add_node("process_values", process_values)

graph.set_entry_point("process_values")
graph.set_finish_point("process_values")

app = graph.compile()

input = {
    "values": [1, 2, 3, 4, 5, 6],
    "name": "Steve"
}

result = app.invoke(input)
print(result)

{'values': [1, 2, 3, 4, 5, 6], 'name': 'Steve'}
{'values': [1, 2, 3, 4, 5, 6], 'name': 'Steve', 'result': 'Hi there, Steve! Your sum is 21'}
{'values': [1, 2, 3, 4, 5, 6], 'name': 'Steve', 'result': 'Hi there, Steve! Your sum is 21'}


In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    name: str
    age: int
    final: str

def first_node(state: AgentState) -> AgentState:

    state["final"] = f"Hi {state["name"]}. "
    return state

def second_node(state: AgentState) -> AgentState:

    state["final"] = state["final"] + f"You are {state['age']} years old"
    return state

graph = StateGraph(AgentState)

graph.add_node("first_node", first_node)
graph.add_node("second_node", second_node)

graph.set_entry_point("first_node")
graph.add_edge("first_node", "second_node")
graph.set_finish_point("second_node")
app = graph.compile()

res = app.invoke({"name": "Steve", "age": 2})
print(res)

{'name': 'Steve', 'age': 2, 'final': 'Hi Steve. You are 2 years old'}


In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    number1: int
    number2: int
    operation: str
    result: int

def adder(state: AgentState) -> AgentState:
    state["result"] = state["number1"] + state["number2"]

    return state

def subtractor(state: AgentState) -> AgentState:
    state["result"] = state["number1"] - state["number2"]

    return state

def operation_router(state: AgentState) -> AgentState:

    if state["operation"] == "+":
        return "addition"
    elif state["operation"] == "-":
        return "subtraction"

graph = StateGraph(AgentState)

graph.add_node("addition", adder)
graph.add_node("subtraction", subtractor)
graph.add_node("router", lambda state:state) # passthrough function

graph.add_edge(START, "router")
graph.add_conditional_edges(
    "router",
    operation_router,

    {
        # Edge: Node
        "addition": "addition",
        "subtraction": "subtraction"
    }
)

graph.add_edge("addition", END)
graph.add_edge("subtraction", END)

app = graph.compile()
res = app.invoke({"number1": 2, "number2": 2, "operation": "-"})
print(res)

{'number1': 2, 'number2': 2, 'operation': '-', 'result': 0}


In [20]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
import random

class AgentState(TypedDict):
    name: str
    number: List[int]
    counter: int

def greeting(state: AgentState) -> AgentState:
    state["name"] = f"Hi there, {state["name"]}"
    state["counter"] = 0
    return state

def random_node(state: AgentState) -> AgentState:
    state["number"].append(random.randint(0,10))
    state["counter"] += 1
    return state

def should_continue(state: AgentState) -> AgentState:

    if state["counter"] <= 5:
        print("inside loop")
        return "loop"
    else:
        return "exit"

graph = StateGraph(AgentState)

graph.add_node("greeting", greeting)
graph.add_node("random_node", random_node)

graph.add_edge(START, "greeting")
graph.add_edge("greeting", "random_node")
graph.add_conditional_edges(
    "random_node",
    should_continue,
    {
        "loop": "random_node",
        "exit": END
    }
)

app = graph.compile()
app.invoke({"name": "vro", "number": []})

inside loop
inside loop
inside loop
inside loop
inside loop


{'name': 'Hi there, vro', 'number': [3, 4, 7, 9, 5, 6], 'counter': 6}